In [1]:
# %% P6 pre-step: list columns of the files Phase 6 will join (read-only)
from pathlib import Path
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 80)

ROOT = Path(r"D:\Siam\Thesis_GeoAI")
files = {
    "phase5": ROOT / "05_models" / "phase5_v01" / "phase5_risk_v01.csv",
    "phase4": ROOT / "05_models" / "phase4_v01" / "phase4_shap_ensemble_v01.csv",
    "v03": ROOT / "05_models" / "final_v03" / "hazard_predictions_v03.csv",
    "v02": ROOT / "05_models" / "final_v02" / "hazard_predictions_v02.csv",
    "diag_upz": ROOT / "05_models" / "phase4_v01" / "diag_upazila_v01.csv",
    "obs_mask": ROOT / "03_features" / "Observability_Mask_v01.csv",
}
for name, p in files.items():
    print(f"\n===== {name}: {p.name} | exists: {p.exists()} =====")
    if not p.exists():
        continue
    df = pd.read_csv(p)
    print("rows:", len(df), "| cols:", len(df.columns))
    print("columns:", list(df.columns))
    if "Grid_ID" in df.columns:
        print("Grid_ID unique:", df["Grid_ID"].is_unique)
    print(df.head(2).to_string())
print("\nP6 PRE-STEP OK")


===== phase5: phase5_risk_v01.csv | exists: True =====
rows: 160874 | cols: 27
columns: ['Grid_ID', 'Tide_Exceed_m', 'H_Flood', 'H_Wind', 'H_Tide', 'H_Index', 'E_Index', 'S_Index', 'AC_Index', 'V_Index', 'Risk_Index', 'Risk_Rank_Pct', 'Risk_Class', 'Is_Top10_Risk', 'Risk_MC_Rank_Mean_Pct', 'Risk_MC_Rank_P05_Pct', 'Risk_MC_Rank_P95_Pct', 'Risk_Top10_Prob_Pct', 'Hazard_Prob_NoCrop', 'Risk_Rank_Pct_SSP245_2050', 'Risk_Class_SSP245_2050', 'Risk_Rank_Pct_SSP585_2050', 'Risk_Class_SSP585_2050', 'Risk_Rank_Pct_SSP245_2100', 'Risk_Class_SSP245_2100', 'Risk_Rank_Pct_SSP585_2100', 'Risk_Class_SSP585_2100']
Grid_ID unique: True
   Grid_ID  Tide_Exceed_m   H_Flood  H_Wind    H_Tide   H_Index  E_Index   S_Index  AC_Index   V_Index  Risk_Index  Risk_Rank_Pct  Risk_Class  Is_Top10_Risk  Risk_MC_Rank_Mean_Pct  Risk_MC_Rank_P05_Pct  Risk_MC_Rank_P95_Pct  Risk_Top10_Prob_Pct  Hazard_Prob_NoCrop  Risk_Rank_Pct_SSP245_2050  Risk_Class_SSP245_2050  Risk_Rank_Pct_SSP585_2050  Risk_Class_SSP585_2050  Risk_R

In [2]:
# %% Cell 1: settings, load, join, reproduce Phase 5 risk exactly
from pathlib import Path
import json
import numpy as np
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 80)
 
ROOT = Path(r"D:\Siam\Thesis_GeoAI")
OUT = ROOT / "05_models" / "phase6_v01"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)
 
F_MASTER = ROOT / "04_master_table" / "master_v06_20261020.csv"
F_P5 = ROOT / "05_models" / "phase5_v01" / "phase5_risk_v01.csv"
F_P4 = ROOT / "05_models" / "phase4_v01" / "phase4_shap_ensemble_v01.csv"
F_V03 = ROOT / "05_models" / "final_v03" / "hazard_predictions_v03.csv"
F_V02 = ROOT / "05_models" / "final_v02" / "hazard_predictions_v02.csv"
F_UPZ = ROOT / "05_models" / "phase4_v01" / "diag_upazila_v01.csv"
F_GRID = ROOT / "02_boundary_grid" / "grid_250m_v01.gpkg"
 
# ---- Pre-specified decision rules (do not change after seeing results) ----
CELL_AREA_KM2 = 0.0625
MONITOR_PCT = 80.0             # Monitor = risk percentile 80-90 (top 10-20%), not already top 10%
UNDER_PRED_RATIO = 1.5         # upazila observed / OOF-predicted flood rate >= 1.5 -> possible under-prediction
OVER_PRED_RATIO = 1 / 1.5      # <= 0.667 -> possible over-prediction (reported, not reclassified)
DRIVER_STABLE_PCT = 80.0       # Driver_Stability_Pct below this -> lower driver confidence
ACCESS_GAP_AC = 0.25           # AC_Index <= 0.25 (lowest access quartile) -> add access measure
HIDDEN_V02_MAX = 0.20          # hidden hazard: v02 flood percentile < 0.20 (Very Low quintile)
HIDDEN_V03_MIN = 0.60          # ... and v03 flood percentile >= 0.60 (top two quintiles)
 
mcols = ["Grid_ID", "District", "Upazila_Name", "Is_Water", "Population_Count", "Building_Count",
         "Poverty_Rate_Upazila_Pct", "Distance_to_Hospital_m", "Distance_to_Road_m"]
m = pd.read_csv(F_MASTER, usecols=mcols)
m = m[m["Is_Water"] == 0].drop(columns="Is_Water")
assert len(m) == 160874 and m["Grid_ID"].is_unique
 
p5 = pd.read_csv(F_P5)
p4 = pd.read_csv(F_P4)
v03 = pd.read_csv(F_V03).drop(columns="District")
v02 = pd.read_csv(F_V02)[["Grid_ID", "Hazard_Prob_LGBM"]].rename(columns={"Hazard_Prob_LGBM": "Hazard_Prob_LGBM_v02"})
 
d = (m.merge(p5, on="Grid_ID", how="inner", validate="one_to_one")
      .merge(p4, on="Grid_ID", how="inner", validate="one_to_one")
      .merge(v03, on="Grid_ID", how="inner", validate="one_to_one")
      .merge(v02, on="Grid_ID", how="inner", validate="one_to_one"))
assert len(d) == 160874, f"join lost rows: {len(d)}"
 
need = ["H_Flood", "H_Wind", "H_Tide", "H_Index", "E_Index", "V_Index", "AC_Index", "Risk_Index", "Risk_Rank_Pct",
        "Is_Top10_Risk", "Risk_Top10_Prob_Pct", "Hazard_Ens_Rank_SD_Pct", "Driver_Stability_Pct", "Is_Within_AOA",
        "Hazard_Prob_LGBM", "Hazard_Score_FR", "Hazard_Prob_LGBM_v02", "Unobservable_Reason", "Population_Count",
        "Poverty_Rate_Upazila_Pct"]
nan = d[need].isna().sum()
assert nan.sum() == 0, f"NaN in required columns: {nan[nan > 0].to_dict()}"
 
# Percentile convention: find the one Phase 5 used, so v02/FR alternatives are built the same way
def pct_avg(x):
    return x.rank(method="average", pct=True)
def pct_mid(x):
    return (x.rank(method="average") - 0.5) / len(x)
err = {"avg": (pct_avg(d["Hazard_Prob_LGBM"]) - d["H_Flood"]).abs().max(),
       "mid": (pct_mid(d["Hazard_Prob_LGBM"]) - d["H_Flood"]).abs().max()}
print("H_Flood reproduction max abs error:", {k: f"{v:.2e}" for k, v in err.items()})
CONV = min(err, key=err.get)
assert err[CONV] < 1e-6, "Cannot reproduce H_Flood from Hazard_Prob_LGBM - paste this output to Claude"
pct = pct_avg if CONV == "avg" else pct_mid
print("percentile convention used by Phase 5:", CONV)
 
# Reproduce the Phase 5 aggregation exactly
H_chk = (d["H_Flood"] + d["H_Wind"] + d["H_Tide"]) / 3
R_chk = H_chk * d["E_Index"] * d["V_Index"]
assert np.allclose(H_chk, d["H_Index"], atol=1e-9), "H_Index is not the equal-weight mean"
assert np.allclose(R_chk, d["Risk_Index"], atol=1e-9), "Risk_Index is not H x E x V"
N_TOP = int(d["Is_Top10_Risk"].sum())
print("top-10% risk cells (Phase 5):", N_TOP, "| min Risk_Rank_Pct inside:", round(d.loc[d["Is_Top10_Risk"] == 1, "Risk_Rank_Pct"].min(), 3))
print("Unobservable_Reason values:", d["Unobservable_Reason"].value_counts().to_dict())
print("Dominant_Increasing_Driver values:", d["Dominant_Increasing_Driver"].value_counts(dropna=False).to_dict())
print("\nCELL 1 OK")

H_Flood reproduction max abs error: {'avg': '3.11e-06', 'mid': '1.11e-16'}
percentile convention used by Phase 5: mid
top-10% risk cells (Phase 5): 16088 | min Risk_Rank_Pct inside: 90.0
Unobservable_Reason values: {'Tree_Canopy': 74500, 'Observable': 56174, 'Mangrove': 29017, 'Tree_And_Mangrove': 1183}
Dominant_Increasing_Driver values: {'Elevation_Mean_m': 40042, 'Distance_to_River_m': 39179, nan: 37052, 'Slope_Mean_deg': 24314, 'Cropland_Fraction': 20287}

CELL 1 OK


In [3]:
# %% Cell 2: priority classes (uncertainty-aware) + upazila fit flag
# Upazila model fit (observed vs OOF-predicted flood rate, from Phase 4 diagnostic)
upz = pd.read_csv(F_UPZ)[["District", "Upazila_Name", "Obs_to_Pred", "Train_Rows"]]
d = d.merge(upz.rename(columns={"Obs_to_Pred": "Upazila_Obs_to_Pred", "Train_Rows": "Upazila_Train_Rows"}),
            on=["District", "Upazila_Name"], how="left", validate="many_to_one")
assert d["Upazila_Obs_to_Pred"].notna().all(), "some upazila names did not match diag_upazila_v01.csv"
d["Upazila_Fit_Flag"] = np.select(
    [d["Upazila_Obs_to_Pred"] >= UNDER_PRED_RATIO, d["Upazila_Obs_to_Pred"] <= OVER_PRED_RATIO],
    ["Under-predicted", "Over-predicted"], default="Consistent")
 
# masks are built AFTER the merge so their index matches d
sd_med = d["Hazard_Ens_Rank_SD_Pct"].median()
top = d["Is_Top10_Risk"] == 1
in_aoa = d["Is_Within_AOA"] == 1
low_sd = d["Hazard_Ens_Rank_SD_Pct"] <= sd_med
monitor = (~top) & (d["Risk_Rank_Pct"] >= MONITOR_PCT)
print(f"median hazard ensemble rank SD (all model cells): {sd_med:.2f} pct points")
 
populated = d["Population_Count"] > 0
mangrove_extrap = (d["Unobservable_Reason"] == "Mangrove") & (~in_aoa)
data_limited = populated & (~top) & (~monitor) & (mangrove_extrap | (d["Upazila_Fit_Flag"] == "Under-predicted"))
 
d["Priority_Class"] = np.select(
    [top & in_aoa & low_sd, top, monitor, data_limited],
    ["1 Act now", "2 Verify first", "3 Monitor", "4 Data-limited / possible under-prediction"],
    default="5 Lower priority")
d["Priority_Class_Code"] = d["Priority_Class"].str[0].astype(int)
d["Verify_Reason"] = np.where(d["Priority_Class_Code"] == 2,
                              np.select([(~in_aoa) & (~low_sd), ~in_aoa, ~low_sd],
                                        ["Outside AOA + high SD", "Outside AOA", "High SD"], default=""), "")
 
pop_total = d["Population_Count"].sum()
summ = d.groupby("Priority_Class").agg(
    Cells=("Grid_ID", "size"), Pop_k=("Population_Count", lambda s: s.sum() / 1e3),
    Within_AOA_Pct=("Is_Within_AOA", lambda s: s.mean() * 100),
    Mean_Top10_Prob_Pct=("Risk_Top10_Prob_Pct", "mean"),
    Mean_Poverty_Pct_PopW=("Poverty_Rate_Upazila_Pct", lambda s: np.average(s, weights=d.loc[s.index, "Population_Count"])
                           if d.loc[s.index, "Population_Count"].sum() > 0 else np.nan))
summ.insert(1, "Area_km2", summ["Cells"] * CELL_AREA_KM2)
summ.insert(3, "Pop_Share_Pct", summ["Pop_k"] * 1e3 / pop_total * 100)
print("\n== Priority classes ==\n", summ.round(1).to_string())
assert summ.loc[["1 Act now", "2 Verify first"], "Cells"].sum() == N_TOP
 
act = d["Priority_Class_Code"] == 1
print(f"\nAct now cells robust in MC (Top10_Prob >= 80%): {(d.loc[act, 'Risk_Top10_Prob_Pct'] >= 80).mean() * 100:.1f}%")
print("Verify first reasons (cells):", d.loc[d["Priority_Class_Code"] == 2, "Verify_Reason"].value_counts().to_dict())
 
by_dist = d.pivot_table(index="District", columns="Priority_Class", values="Population_Count", aggfunc="sum", fill_value=0) / 1e3
print("\n== Population (k) by district and class ==\n", by_dist.round(1).to_string())
 
fit_top = d[top].groupby("Upazila_Fit_Flag")["Population_Count"].sum() / 1e3
print("\nTop-10% population (k) by upazila fit flag:", fit_top.round(1).to_dict())
print("Under-predicted upazilas:", sorted(d.loc[d["Upazila_Fit_Flag"] == "Under-predicted", "Upazila_Name"].unique()))
print("Over-predicted upazilas:", sorted(d.loc[d["Upazila_Fit_Flag"] == "Over-predicted", "Upazila_Name"].unique()))
print("\nCELL 2 OK")

median hazard ensemble rank SD (all model cells): 5.35 pct points

== Priority classes ==
                                              Cells  Area_km2   Pop_k  Pop_Share_Pct  Within_AOA_Pct  Mean_Top10_Prob_Pct  Mean_Poverty_Pct_PopW
Priority_Class                                                                                                                                 
1 Act now                                     3295     205.9   273.5            3.5           100.0                 86.2                   32.3
2 Verify first                               12793     799.6  1187.1           15.4            50.3                 85.8                   30.8
3 Monitor                                    16087    1005.4  1145.9           14.8            61.9                 13.5                   28.3
4 Data-limited / possible under-prediction   21330    1333.1  1108.9           14.4            65.4                  0.1                   25.5
5 Lower priority                            1

In [4]:
# %% Cell 3: dominant SHAP driver -> candidate adaptation measure (classes 1-3 only)
# Candidate measure TYPES for planning discussion, not engineering design. SHAP explains the model, not causation.
MEASURES = {
    "Elevation_Mean_m": ("Terrain elevation", "Structural",
                         "Raised homestead plinths; flood/cyclone shelters; embankment height and maintenance"),
    "Distance_to_River_m": ("River proximity", "Structural + nature-based",
                            "Riverbank protection and embankment repair; vegetated riverbank buffers"),
    "Slope_Mean_deg": ("Flat terrain / drainage", "Structural",
                       "Drainage canal re-excavation; sluice-gate maintenance"),
    "Cropland_Fraction": ("Cropland setting (lower confidence)", "Agricultural",
                          "Flood- and salinity-tolerant crop varieties; agricultural early warning"),
    "None": ("No hazard-increasing driver", "Social / access",
             "Risk driven by exposure and vulnerability: early warning, evacuation planning, social protection"),
}
ACCESS_MEASURE = "Access: shelter capacity and all-weather road links to health facilities"
 
prio = d["Priority_Class_Code"] <= 3
drv = d["Dominant_Increasing_Driver"].fillna("None")
assert set(drv.unique()) <= set(MEASURES), f"unexpected driver values: {set(drv.unique()) - set(MEASURES)}"
d["Access_Gap"] = (d["AC_Index"] <= ACCESS_GAP_AC).astype(int)
d["Driver_Label"] = np.where(prio, drv.map(lambda k: MEASURES[k][0]), "")
d["Measure_Type"] = np.where(prio, drv.map(lambda k: MEASURES[k][1]), "")
d["Measure_Primary"] = np.where(prio, drv.map(lambda k: MEASURES[k][2]), "")
d["Measure_Access"] = np.where(prio & (d["Access_Gap"] == 1), ACCESS_MEASURE, "")
 
reasons = pd.DataFrame({
    "Cropland driver": drv == "Cropland_Fraction",
    "Unstable driver": d["Driver_Stability_Pct"] < DRIVER_STABLE_PCT,
    "Outside AOA": ~in_aoa})
d["Driver_Confidence"] = np.where(
    ~prio, "",
    np.where(reasons.any(axis=1), "Lower: " + reasons.apply(lambda r: ", ".join(r.index[r.values]), axis=1), "Higher"))
 
tab = (d[prio].assign(Driver=drv[prio])
       .groupby(["Priority_Class", "Driver"])
       .agg(Cells=("Grid_ID", "size"), Pop_k=("Population_Count", lambda s: s.sum() / 1e3),
            Access_Gap_Pct=("Access_Gap", lambda s: s.mean() * 100),
            Higher_Conf_Pct=("Driver_Confidence", lambda s: (s == "Higher").mean() * 100)))
print("== Driver -> measure, classes 1-3 ==\n", tab.round(1).to_string())
print("\nDriver_Confidence (classes 1-3, cells):", d.loc[prio, "Driver_Confidence"].str.split(":").str[0].value_counts().to_dict())
print(f"Act now population with access gap: {d.loc[act & (d['Access_Gap'] == 1), 'Population_Count'].sum() / 1e3:.1f}k")
print("\nCELL 3 OK")

== Driver -> measure, classes 1-3 ==
                                     Cells  Pop_k  Access_Gap_Pct  Higher_Conf_Pct
Priority_Class Driver                                                            
1 Act now      Cropland_Fraction      254   16.7            21.7              0.0
               Distance_to_River_m    222   26.9            12.2             75.7
               Elevation_Mean_m      2199  169.9             9.6             78.9
               None                   316   37.4            13.6             81.6
               Slope_Mean_deg         304   22.6            12.8             53.9
2 Verify first Cropland_Fraction      239   14.1            36.4              0.0
               Distance_to_River_m   2677  283.2            15.8             25.5
               Elevation_Mean_m      6936  612.3            16.0             40.5
               None                  1170  114.9            23.5             21.7
               Slope_Mean_deg        1771  162.6            

In [5]:
# %% Cell 4: decision impact (v02 vs v03 hazard, ML vs FR) + equity lens + hidden-risk cells
def risk_with_flood(h_flood_pct):
    """Same Phase 5 aggregation, only the flood-hazard percentile is swapped."""
    return ((h_flood_pct + d["H_Wind"] + d["H_Tide"]) / 3) * d["E_Index"] * d["V_Index"]
 
def top_n(r):
    return r.rank(method="first", ascending=False) <= N_TOP
 
H_v02 = pct(d["Hazard_Prob_LGBM_v02"])
H_fr = pct(d["Hazard_Score_FR"])
alts = {"v02 naive labels (LGBM)": risk_with_flood(H_v02), "v03 Frequency Ratio": risk_with_flood(H_fr)}
top_v03 = top
rows = []
for name, r in alts.items():
    t = top_n(r)
    rows.append({"Comparison": name,
                 "Spearman_Risk_vs_v03_ML": d["Risk_Index"].corr(r, method="spearman"),
                 "Top10_Overlap_Pct": (t & top_v03).sum() / N_TOP * 100,
                 "Pop_Top10_Alt_k": d.loc[t, "Population_Count"].sum() / 1e3,
                 "Pop_Only_In_v03_ML_k": d.loc[top_v03 & ~t, "Population_Count"].sum() / 1e3,
                 "Pop_Only_In_Alt_k": d.loc[t & ~top_v03, "Population_Count"].sum() / 1e3})
    if name.startswith("v02"):
        d["Is_Top10_Risk_v02Hazard"] = t.astype(int)
    else:
        d["Is_Top10_Risk_FRHazard"] = t.astype(int)
impact = pd.DataFrame(rows)
print("== Decision impact ==\n", impact.round(3).to_string(index=False))
 
def popw_pov(mask):
    w = d.loc[mask, "Population_Count"]
    return np.average(d.loc[mask, "Poverty_Rate_Upazila_Pct"], weights=w) if w.sum() > 0 else np.nan
 
t02 = d["Is_Top10_Risk_v02Hazard"] == 1
groups = {"All model cells": pd.Series(True, index=d.index), "Top-10% (v03 ML)": top_v03,
          "Revealed by v03 (top-10% v03, not v02)": top_v03 & ~t02, "Dropped by v03 (top-10% v02, not v03)": t02 & ~top_v03}
eq = pd.DataFrame({g: {"Cells": int(mk.sum()), "Pop_k": d.loc[mk, "Population_Count"].sum() / 1e3,
                       "Upazila_Poverty_Pct_PopW": popw_pov(mk),
                       "Unobservable_Cells_Pct": (d.loc[mk, "Is_Observable"] == 0).mean() * 100}
                   for g, mk in groups.items()}).T
print("\n== Equity lens (upazila-level poverty, population-weighted) ==\n", eq.round(1).to_string())
 
d["Is_Hidden_Hazard"] = ((H_v02 < HIDDEN_V02_MAX) & (d["H_Flood"] >= HIDDEN_V03_MIN)).astype(int)
hid = d["Is_Hidden_Hazard"] == 1
print(f"\nHidden-hazard cells (v02 Very Low, v03 top two quintiles): {hid.sum()} cells, "
      f"{hid.sum() * CELL_AREA_KM2:.0f} km2, {d.loc[hid, 'Population_Count'].sum() / 1e3:.1f}k people "
      f"({d.loc[hid, 'Population_Count'].sum() / pop_total * 100:.1f}%), unobservable {(d.loc[hid, 'Is_Observable'] == 0).mean() * 100:.1f}%, "
      f"pop-weighted upazila poverty {popw_pov(hid):.1f}% (all cells {popw_pov(pd.Series(True, index=d.index)):.1f}%)")
hid_dist = d[hid].groupby("District").agg(Cells=("Grid_ID", "size"), Pop_k=("Population_Count", lambda s: s.sum() / 1e3))
print(hid_dist.round(1).to_string())
print("\nCELL 4 OK")

== Decision impact ==
              Comparison  Spearman_Risk_vs_v03_ML  Top10_Overlap_Pct  Pop_Top10_Alt_k  Pop_Only_In_v03_ML_k  Pop_Only_In_Alt_k
v02 naive labels (LGBM)                    0.985             79.289         1341.181               324.509            205.059
    v03 Frequency Ratio                    0.991             87.823         1434.787               175.825            149.980

== Equity lens (upazila-level poverty, population-weighted) ==
                                            Cells   Pop_k  Upazila_Poverty_Pct_PopW  Unobservable_Cells_Pct
All model cells                         160874.0  7725.1                      23.2                    65.1
Top-10% (v03 ML)                         16088.0  1460.6                      31.1                    72.1
Revealed by v03 (top-10% v03, not v02)    3332.0   324.5                      32.2                   100.0
Dropped by v03 (top-10% v02, not v03)     3332.0   205.1                      28.7                    40.5

In [6]:
# %% Cell 5: maps (cell centroids) + save outputs
import geopandas as gpd
import matplotlib.pyplot as plt
 
g = gpd.read_file(F_GRID)
assert "Grid_ID" in g.columns, f"grid columns: {list(g.columns)}"
assert g.crs is not None and g.crs.to_epsg() == 32646, f"grid CRS: {g.crs}"
cent = g[["Grid_ID"]].copy()
cent["X"] = g.geometry.centroid.x
cent["Y"] = g.geometry.centroid.y
mp = d.merge(cent, on="Grid_ID", how="left", validate="one_to_one")
assert mp["X"].notna().all(), "some Grid_ID missing in grid gpkg"
 
CLASS_COLORS = {1: "#b2182b", 2: "#ef8a62", 3: "#fddbc7", 4: "#6a51a3", 5: "#d9d9d9"}
CLASS_NAMES = dict(zip(d["Priority_Class_Code"], d["Priority_Class"]))
fig, ax = plt.subplots(1, 2, figsize=(16, 9))
for code in [5, 4, 3, 2, 1]:
    s = mp[mp["Priority_Class_Code"] == code]
    ax[0].scatter(s["X"], s["Y"], s=0.4, marker="s", linewidths=0, c=CLASS_COLORS[code], label=CLASS_NAMES.get(code, str(code)))
ax[0].set_title("Adaptation priority classes (250 m)")
ax[0].legend(markerscale=12, loc="lower left", fontsize=8)
 
bg = mp[mp["Is_Hidden_Hazard"] == 0]
ax[1].scatter(bg["X"], bg["Y"], s=0.4, marker="s", linewidths=0, c="#e0e0e0")
hs = mp[mp["Is_Hidden_Hazard"] == 1]
ax[1].scatter(hs["X"], hs["Y"], s=0.4, marker="s", linewidths=0, c="#2166ac", label="Hidden hazard (v02 Very Low, v03 top 40%)")
ax[1].set_title("Hidden flood hazard under naive SAR labels")
ax[1].legend(markerscale=12, loc="lower left", fontsize=8)
for a in ax:
    a.set_aspect("equal"); a.set_xticks([]); a.set_yticks([])
plt.tight_layout()
fig.savefig(FIG / "Fig_P6_Map_Priority_Hidden.png", dpi=200)
plt.close(fig)
 
new_cols = ["Grid_ID", "Priority_Class", "Priority_Class_Code", "Verify_Reason", "Upazila_Obs_to_Pred", "Upazila_Fit_Flag",
            "Access_Gap", "Driver_Label", "Measure_Type", "Measure_Primary", "Measure_Access", "Driver_Confidence",
            "Is_Top10_Risk_v02Hazard", "Is_Top10_Risk_FRHazard", "Is_Hidden_Hazard"]
d[new_cols].to_csv(OUT / "phase6_priority_v01.csv", index=False)
summ.to_csv(OUT / "priority_summary_v01.csv")
by_dist.to_csv(OUT / "priority_pop_by_district_v01.csv")
tab.to_csv(OUT / "driver_measure_v01.csv")
impact.to_csv(OUT / "decision_impact_v01.csv", index=False)
eq.to_csv(OUT / "equity_lens_v01.csv")
cfg = {"MONITOR_PCT": MONITOR_PCT, "UNDER_PRED_RATIO": UNDER_PRED_RATIO, "OVER_PRED_RATIO": OVER_PRED_RATIO,
       "DRIVER_STABLE_PCT": DRIVER_STABLE_PCT, "ACCESS_GAP_AC": ACCESS_GAP_AC, "HIDDEN_V02_MAX": HIDDEN_V02_MAX,
       "HIDDEN_V03_MIN": HIDDEN_V03_MIN, "SD_MEDIAN_PCT": float(sd_med), "N_TOP": N_TOP, "PCT_CONVENTION": CONV,
       "MEASURES": MEASURES, "ACCESS_MEASURE": ACCESS_MEASURE}
(OUT / "phase6_config_v01.json").write_text(json.dumps(cfg, indent=2))
print("Saved to", OUT, "->", sorted(p.name for p in OUT.glob("*.*")), "| figure:", [p.name for p in FIG.glob("*.png")])
print("\nCELL 5 OK")

Saved to D:\Siam\Thesis_GeoAI\05_models\phase6_v01 -> ['decision_impact_v01.csv', 'driver_measure_v01.csv', 'equity_lens_v01.csv', 'phase6_config_v01.json', 'phase6_priority_v01.csv', 'priority_pop_by_district_v01.csv', 'priority_summary_v01.csv'] | figure: ['Fig_P6_Map_Priority_Hidden.png']

CELL 5 OK


In [7]:
# %% Cell 6: summary to paste back
print("=== PHASE 6 SUMMARY ===")
print(summ[["Cells", "Area_km2", "Pop_k", "Pop_Share_Pct", "Within_AOA_Pct"]].round(1).to_string())
print(impact[["Comparison", "Top10_Overlap_Pct", "Pop_Only_In_v03_ML_k", "Pop_Only_In_Alt_k"]].round(1).to_string(index=False))
print(eq[["Pop_k", "Upazila_Poverty_Pct_PopW"]].round(1).to_string())
print(f"Hidden-hazard population: {d.loc[hid, 'Population_Count'].sum() / 1e3:.1f}k")
print("\nCELL 6 OK")

=== PHASE 6 SUMMARY ===
                                             Cells  Area_km2   Pop_k  Pop_Share_Pct  Within_AOA_Pct
Priority_Class                                                                                     
1 Act now                                     3295     205.9   273.5            3.5           100.0
2 Verify first                               12793     799.6  1187.1           15.4            50.3
3 Monitor                                    16087    1005.4  1145.9           14.8            61.9
4 Data-limited / possible under-prediction   21330    1333.1  1108.9           14.4            65.4
5 Lower priority                            107369    6710.6  4009.6           51.9            48.9
             Comparison  Top10_Overlap_Pct  Pop_Only_In_v03_ML_k  Pop_Only_In_Alt_k
v02 naive labels (LGBM)               79.3                 324.5              205.1
    v03 Frequency Ratio               87.8                 175.8              150.0
                        